In [ ]:
 using LinearAlgebra
 using DifferentialEquations
 using BSplineKit
 using Plots

In [ ]:
function sol_baseflowODE(tspan,Num)

    function oneDiskODE!(du, u , p, t)
        
        U = u[1]
        dU = u[2]
        V = u[3]
        dV = u[4]
        W = u[5]
        du[1] = dU
        ddU = U^2 + W*dU - (V+1.0e0)^2
        du[2] = ddU
        ddV = 2.0e0*U*(V + 1.0e0) + W*dV
        du[3] = dV
        du[4] = ddV                          
        du[5] = -2.0e0*U

    end
    function oneDiskbc!(residual, u , p, t)

        residual[1] = u[begin][1] 
        residual[2] = u[begin][3] 
        residual[3] = u[begin][5] 
        residual[4] = u[end][1] 

        residual[5] = u[end][3] + 1.0e0
    end
        prob = BVProblem(oneDiskODE!, oneDiskbc!, [0.0, 0.5103341351120374, 0.0, -0.6151547026271073, 0.0] ,tspan, dtmax=0.01)
        sol = solve(prob, Shooting(Vern7()), dt=0.01)
        t=range(0.0, 20, Num)
        u=sol(t)
    
    return u , t

end

In [ ]:
function velocity(u,phi)

    U = u[1 , :]
    dU = u[2 , :]
    V = u[3 , :]
    dV = u[4 , :]
    W = u[5 , :]
    F_U = itp = interpolate(t, U , BSplineOrder(4))
    F_dU = itp = interpolate(t, dU , BSplineOrder(4))
    F_dV = itp = interpolate(t, dV , BSplineOrder(4))
    F_W= itp = interpolate(t, W , BSplineOrder(4))
    F_phi = interpolate(t, PHI , BSplineOrder(4))
    return U,dU,V,dV,W,F_U,F_dU,F_dV,F_W,F_phi

end

In [ ]:
N = 1001
tspan = (0,20)
t = range(0,20,N)
sigma = 0.7
Tw = 0.5
gamma = 1.4
u,z = sol_baseflowODE(tspan,N)
PHI = phi_var(u0,0.02,N)
u0,du0,v0,dv0,w0,F_u,F_du,F_dv,F_w,F_phi = velocity(u,PHI)

In [ ]:
function f_q(sigma,gamma,Tw,F_du,F_dv,F_u,F_phi)
    function ODE_f!(du,u,p,t)
        q = u[1]
        dq = u[2]
        du[1] = dq
        du[2] = 2*sigma*(F_du(t)^2+F_dv(t)^2+F_u(t)*u[1]-F_phi(t)*u[2])
    end
    function BC_f!(residual, u, p, t)
        residual[1] = u[begin][1]
        residual[2] = u[end][1]
    end     
    prob = BVProblem(ODE_f!, BC_f!, [0.0, 0.0], tspan)
    sol = solve(prob, Shooting(Vern7()), dt=0.01)
    f = sol(t)
    function ODE_q!(du,u,p,t)
        q = u[1]
        dq = u[2]
        du[1] = dq
        du[2] = -2*sigma*F_phi(t)*u[2]
    end
    function BC_q!(residual, u, p, t)
        residual[1] = u[begin][1] - 1
        residual[2] = u[end][1]
    end     
    prob = BVProblem(ODE_q!, BC_q!, [1,0], tspan)
    sol1 = solve(prob,MIRK4(), dt=0.01)
    q = sol1(t)
    f = f[1,:]
    q = q[1,:]
    return f,q
 end
function T_var(Mx,f,q,Tw)
    T = 1 .- ( (gamma-1)/2 )*Mx^2 * f + (Tw - 1) * q
    return T
 end

T_var (generic function with 1 method)

In [ ]:
f,q = f_q(sigma,gamma,Tw,F_du,F_dv,F_u,F_phi)

UndefVarError: UndefVarError: `Mx` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [122]:
Mx = 6
T = T_var(Mx,f,q,Tw)

1001-element Vector{Float64}:
 0.4999999999999999
 0.5677004814612147
 0.6329021693558388
 0.6956830438879917
 0.7561184800350631
 0.8142813583061248
 0.8702421692451201
 0.9240691119506845
 0.9758281874670496
 1.0255832869550423
 ⋮
 0.9999254900351638
 0.99992530913012
 0.9999251304512574
 0.9999249539711813
 0.9999247796628341
 0.9999246074994912
 0.9999244374547571
 0.999924269502561
 0.9999241036171523